# 📦 Amazon Orders Data Pipeline (2020–2024)
## 🧩 Notebook 01 — Data Ingestion & Validation

---

## 🔍 Project Overview

This project analyzes Amazon transactional order data spanning **January 1, 2020 to December 29, 2024**.  
The dataset contains transaction-level information including:

- Order details
- Customer information
- Product attributes
- Pricing & discounts
- Taxes & shipping costs
- Payment methods
- Order status
- Geographic information

The overall objective of this project is to extract actionable business insights related to:

- 📈 Sales performance trends  
- 🛍️ Product & category performance  
- 👥 Customer purchasing behavior  
- 💳 Payment method usage  
- 🚚 Shipping & tax impact  
- 🌍 Regional distribution  
- 🏷️ Pricing and discount patterns  

---

## 🏗 Role of This Notebook in the Pipeline

This notebook is responsible for:

1. Initializing the data processing environment
2. Enforcing schema consistency
3. Ingesting the raw dataset
4. Performing structured data validation checks
5. Preparing a clean base DataFrame for downstream analysis

This notebook must be executed before running:

- 02_data_cleaning  
- 03_exploratory_data_analysis  
- 04_feature_engineering  
- 05_statistical_testing  
- 06_ml_modeling  

It serves as the **foundation layer** of the end-to-end analytics pipeline.

---

## 🛠 Tools & Technologies

The following technologies are used across the project:

- **NumPy** – Numerical computations  
- **Pandas** – Data manipulation  
- **Matplotlib & Seaborn** – Data visualization  
- **PySpark** – Distributed data processing  
- **Scikit-learn** – Feature scaling & modeling  
- **SciPy & Statsmodels** – Statistical analysis  

PySpark enables scalable ingestion and validation of large transactional datasets, while Python-based libraries support statistical modeling and visualization in later stages.

---

> 📌 Note: This notebook establishes the trusted base dataset for all downstream analytics, feature engineering, and machine learning workflows.

## 🔧 Environment Initialization

This notebook initializes:
- Spark session
- Required Python libraries
- Shared schema definitions

It acts as the foundational setup notebook for all subsequent analysis notebooks.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as f
import scipy.stats as stats
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [0]:
spark = SparkSession.builder \
    .appName("amazon") \
    .getOrCreate()


### 🏗 Schema Enforcement Strategy

An explicit schema is defined using `StructType` to ensure:

- Data type consistency
- Prevention of incorrect inference
- Improved performance
- Stronger validation controls

Schema enforcement prevents silent data corruption during ingestion.

In [0]:
order_schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("OrderDate", DateType(), True),       
    StructField("CustomerID", StringType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Brand", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Tax", DoubleType(), True),
    StructField("ShippingCost", DoubleType(), True),
    StructField("TotalAmount", DoubleType(), True),
    StructField("PaymentMethod", StringType(), True),
    StructField("OrderStatus", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("SellerID", StringType(), True),
])


In [0]:
df = spark.read \
    .option("header","true") \
    .schema(order_schema) \
    .option("inferSchema","false")\
    .option("delimiter",",") \
    .csv("/Volumes/main/default/datasets/Amazon.csv")

In [0]:
print(f"\nRows : {df.count()} , Columns : {len(df.columns)}\n")


Rows : 100000 , Columns : 20



## 🔎 Data Validation Framework

The following validation checks are performed:

1. Structural Validation
   - Schema verification
   - Column count validation

2. Completeness Checks
   - Null value assessment
   - Missing critical fields detection

3. Uniqueness Checks
   - Duplicate OrderID detection

4. Range & Constraint Checks
   - Quantity > 0
   - Price >= 0
   - Tax >= 0
   - Shipping Cost >= 0

5. Date Validation
   - Valid date parsing
   - Date range within expected bounds

6. Business Rule Validation
   - Order status validity
   - Payment method consistency

Any validation failures will be logged for review.

In [0]:
print("\nAmazon Orders Table Schema\n")
df.printSchema()


Amazon Orders Table Schema

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Tax: double (nullable = true)
 |-- ShippingCost: double (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SellerID: string (nullable = true)



### 🔹Schema Validation

In [0]:
expected_columns = len(order_schema.fields)
actual_columns = len(df.columns)

assert expected_columns == actual_columns, "Column count mismatch!"
print("✔ Column count validation passed.")

✔ Column count validation passed.


### 🔹 Row Count Check

In [0]:
row_count = df.count()
print(f"Total Rows: {row_count}")

assert row_count > 0, "Dataset is empty!"

Total Rows: 100000


### 🔹Null Value Check

In [0]:
df.select([
    f.sum(f.col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
]).display()

OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 🔹Duplicate Check

In [0]:
duplicate_count = (
    df.groupBy("OrderID")
      .count()
      .filter("count > 1")
      .count()
)

print(f"Duplicate OrderIDs: {duplicate_count}")

Duplicate OrderIDs: 0


### 🔹Range & Constraint Checks

In [0]:
# Quantity must be positive
invalid_quantity = df.filter("Quantity <= 0").count()
print(f"Invalid Quantity Records: {invalid_quantity}")

# Price must be non-negative
invalid_price = df.filter("UnitPrice < 0").count()
print(f"Invalid Price Records: {invalid_price}")

# Tax must be non-negative
invalid_tax = df.filter("Tax < 0").count()
print(f"Invalid Tax Records: {invalid_tax}")

Invalid Quantity Records: 0
Invalid Price Records: 0
Invalid Tax Records: 0


### 🔹 Date Range Validation

In [0]:
date_range = df.select(
    f.min("OrderDate").alias("MinDate"),
    f.max("OrderDate").alias("MaxDate")
)

date_range.show()

+----------+----------+
|   MinDate|   MaxDate|
+----------+----------+
|2020-01-01|2024-12-29|
+----------+----------+



### 🔹 Categorical Consistency Check

In [0]:
df.select("OrderStatus").distinct().show()

+-----------+
|OrderStatus|
+-----------+
|    Pending|
|    Shipped|
|   Returned|
|  Delivered|
|  Cancelled|
+-----------+



In [0]:
df.select("PaymentMethod").distinct().show()

+----------------+
|   PaymentMethod|
+----------------+
|      Amazon Pay|
|     Credit Card|
|      Debit Card|
|Cash on Delivery|
|             UPI|
|     Net Banking|
+----------------+



### 🔹Validation Summary Report

In [0]:
validation_results = spark.createDataFrame([
    ("Duplicate Orders", duplicate_count),
    ("Invalid Quantity", invalid_quantity),
    ("Invalid Price", invalid_price)
], ["Check", "IssueCount"])

validation_results.show()

+----------------+----------+
|           Check|IssueCount|
+----------------+----------+
|Duplicate Orders|         0|
|Invalid Quantity|         0|
|   Invalid Price|         0|
+----------------+----------+



In [0]:
display(df.limit(10))

OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0,0.0,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.1,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411
ORD0000006,2022-12-31,CUST042705,Karan Sharma,P00023,Cookware Set,Books,ReadMore,4,449.73,0.0,215.87,2.74,2017.53,UPI,Delivered,Los Angeles,CA,United States,SELL01494
ORD0000007,2024-09-20,CUST037667,Aarav Verma,P00030,Dress Shirt,Clothing,UrbanStyle,2,219.81,0.2,28.14,14.97,394.81,UPI,Delivered,Chicago,IL,Australia,SELL01676
ORD0000008,2022-11-10,CUST031165,Rohit Kumar,P00028,Jeans,Toys & Games,KiddoFun,2,306.51,0.05,29.12,6.24,617.73,Debit Card,Pending,Denver,CO,India,SELL00510
ORD0000009,2024-06-26,CUST026965,Aman Kapoor,P00031,Kids Toy Car,Sports & Outdoors,Apex,4,146.09,0.0,46.75,7.03,638.14,Debit Card,Delivered,Washington,DC,United States,SELL01895
ORD0000010,2020-05-01,CUST029472,Aarav Reddy,P00001,Wireless Earbuds,Clothing,Apex,2,278.21,0.1,60.09,4.88,565.75,Credit Card,Delivered,Houston,TX,United States,SELL01584
